# 01 — Limpeza e entendimento das reuniões

Notebook autocontido para ler, validar, normalizar, deduplicar e auditar as transcrições do Wedjat.

## Decisões tomadas

- A reunião é a unidade de negócio e `ID_MEETING` é seu identificador.
- O NDJSON é percorrido em streaming para limitar o uso de memória.
- A limpeza normaliza somente espaços e quebras de linha; palavras, pontuação e stopwords são preservadas.
- Duplicatas exatas são removidas; IDs iguais com registros diferentes interrompem a execução.
- A saída é publicada atomicamente e nenhuma transcrição é exibida nas análises.
- O chunking será feito depois com o tokenizer do modelo e divisão agrupada por reunião.

In [1]:
from __future__ import annotations
import hashlib, json, os, re, tempfile
from dataclasses import asdict, dataclass
from pathlib import Path
from statistics import mean, median
from typing import Any, Iterator, Sequence

SPEAKER_TURN_PATTERN = re.compile(r'\[LOCUTOR\s+\d+\]:')

class DataPreparationError(ValueError):
    pass

@dataclass(frozen=True)
class PreparationSummary:
    input_records: int
    output_meetings: int
    exact_duplicates_removed: int
    total_transcript_characters: int
    total_speaker_turns: int
    shortest_transcript_characters: int
    longest_transcript_characters: int

def iter_ndjson(path: Path) -> Iterator[tuple[int, dict[str, Any]]]:
    with path.open('r', encoding='utf-8') as source:
        for line_number, raw_line in enumerate(source, 1):
            if not raw_line.strip():
                continue
            try:
                record = json.loads(raw_line)
            except json.JSONDecodeError as exc:
                raise DataPreparationError(f'JSON inválido na linha {line_number}: coluna {exc.colno}.') from exc
            if not isinstance(record, dict):
                raise DataPreparationError(f'A linha {line_number} deve conter um objeto JSON.')
            yield line_number, record

def normalize_transcript(text: str) -> str:
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    lines = [re.sub(r'[ \t]+', ' ', line).strip() for line in text.split('\n')]
    return '\n'.join(line for line in lines if line)

def validate_and_enrich_record(record, line_number, required_fields):
    for field in required_fields:
        if field not in record or record[field] in (None, ''):
            raise DataPreparationError(f'Campo obrigatório {field!r} ausente na linha {line_number}.')
    meeting_id = str(record['ID_MEETING']).strip()
    transcript = record['ANON_TRANSCRICAO']
    if not meeting_id:
        raise DataPreparationError(f'ID_MEETING vazio na linha {line_number}.')
    if not isinstance(transcript, str):
        raise DataPreparationError(f'ANON_TRANSCRICAO deve ser texto na linha {line_number}.')
    transcript = normalize_transcript(transcript)
    if not transcript:
        raise DataPreparationError(f'ANON_TRANSCRICAO vazia após normalização na linha {line_number}.')
    prepared = dict(record)
    prepared.update({
        'ID_MEETING': meeting_id,
        'ANON_TRANSCRICAO': transcript,
        'NUM_CARACTERES_TRANSCRICAO': len(transcript),
        'NUM_TURNOS_TRANSCRICAO': len(SPEAKER_TURN_PATTERN.findall(transcript)),
    })
    return prepared

def record_fingerprint(record):
    canonical = json.dumps(record, ensure_ascii=False, sort_keys=True, separators=(',', ':'))
    return hashlib.sha256(canonical.encode('utf-8')).hexdigest()

def write_json_atomic(path, content):
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(mode='w', encoding='utf-8', newline='\n', dir=path.parent, prefix=f'.{path.name}.', suffix='.tmp', delete=False) as handle:
        json.dump(content, handle, ensure_ascii=False, indent=2)
        handle.write('\n')
        temporary_path = Path(handle.name)
    os.replace(temporary_path, path)

def prepare_meetings(input_path, output_path, report_path, required_fields):
    if not input_path.is_file():
        raise DataPreparationError(f'Arquivo de entrada não encontrado: {input_path}')
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fingerprints = {}
    input_records = output_meetings = duplicates = total_characters = total_turns = 0
    shortest = None
    longest = 0
    temporary_handle = tempfile.NamedTemporaryFile(mode='w', encoding='utf-8', newline='\n', dir=output_path.parent, prefix=f'.{output_path.name}.', suffix='.tmp', delete=False)
    temporary_path = Path(temporary_handle.name)
    try:
        with temporary_handle as destination:
            for line_number, source_record in iter_ndjson(input_path):
                input_records += 1
                prepared = validate_and_enrich_record(source_record, line_number, required_fields)
                meeting_id = prepared['ID_MEETING']
                fingerprint = record_fingerprint(prepared)
                previous = fingerprints.get(meeting_id)
                if previous is not None:
                    if previous == fingerprint:
                        duplicates += 1
                        continue
                    raise DataPreparationError(f'ID_MEETING conflitante na linha {line_number}.')
                fingerprints[meeting_id] = fingerprint
                destination.write(json.dumps(prepared, ensure_ascii=False) + '\n')
                characters = prepared['NUM_CARACTERES_TRANSCRICAO']
                output_meetings += 1
                total_characters += characters
                total_turns += prepared['NUM_TURNOS_TRANSCRICAO']
                shortest = characters if shortest is None else min(shortest, characters)
                longest = max(longest, characters)
        os.replace(temporary_path, output_path)
    except Exception:
        temporary_path.unlink(missing_ok=True)
        raise
    summary = PreparationSummary(input_records, output_meetings, duplicates, total_characters, total_turns, shortest or 0, longest)
    write_json_atomic(report_path, asdict(summary))
    return summary

## Execução

A saída processada continua fora do Git porque contém transcrições. O relatório contém apenas totais seguros.

In [2]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
INPUT_PATH = PROJECT_ROOT / 'data' / 'raw' / 'ANON_transcricao (2).json'
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'meetings.jsonl'
REPORT_PATH = PROJECT_ROOT / 'reports' / 'metrics' / 'data_preparation_summary.json'
summary = prepare_meetings(INPUT_PATH, OUTPUT_PATH, REPORT_PATH, ['ID_MEETING', 'ANON_TRANSCRICAO'])
asdict(summary)

{'input_records': 1174,
 'output_meetings': 1126,
 'exact_duplicates_removed': 48,
 'total_transcript_characters': 44055895,
 'total_speaker_turns': 334763,
 'shortest_transcript_characters': 304,
 'longest_transcript_characters': 185575}

## Auditoria segura

Exibimos o esquema sem valores e estatísticas de tamanho para orientar o próximo chunking.

In [3]:
_, first_record = next(iter_ndjson(INPUT_PATH))
{field: {'type': type(value).__name__, 'is_null': value is None, 'string_length': len(value) if isinstance(value, str) else None} for field, value in first_record.items()}

{'ID_MEETING': {'type': 'str', 'is_null': False, 'string_length': 7},
 'DT_MEETING': {'type': 'str', 'is_null': False, 'string_length': 19},
 'FORMATO_MEETING': {'type': 'str', 'is_null': False, 'string_length': 5},
 'ID_STATUS_MEETING': {'type': 'str', 'is_null': False, 'string_length': 1},
 'STATUS_MEETING': {'type': 'str', 'is_null': False, 'string_length': 9},
 'DURACAO_MEETING': {'type': 'str', 'is_null': False, 'string_length': 8},
 'CODT': {'type': 'str', 'is_null': False, 'string_length': 6},
 'FLG_EXTERNO': {'type': 'bool', 'is_null': False, 'string_length': None},
 'DT_CRIACAO': {'type': 'str', 'is_null': False, 'string_length': 19},
 'ANON_TRANSCRICAO': {'type': 'str', 'is_null': False, 'string_length': 91279},
 'UF': {'type': 'str', 'is_null': False, 'string_length': 2},
 'CNAE': {'type': 'str', 'is_null': False, 'string_length': 7},
 'NOME_UNIDADE': {'type': 'str', 'is_null': False, 'string_length': 11},
 'NOME_SEGMENTO': {'type': 'str', 'is_null': False, 'string_length': 

In [4]:
character_counts, speaker_turn_counts = [], []
for _, record in iter_ndjson(OUTPUT_PATH):
    character_counts.append(record['NUM_CARACTERES_TRANSCRICAO'])
    speaker_turn_counts.append(record['NUM_TURNOS_TRANSCRICAO'])

def percentile(values, probability):
    ordered = sorted(values)
    return ordered[round((len(ordered) - 1) * probability)]

{'meetings': len(character_counts), 'characters_mean': round(mean(character_counts), 2), 'characters_median': median(character_counts), 'characters_p90': percentile(character_counts, .90), 'characters_p95': percentile(character_counts, .95), 'speaker_turns_mean': round(mean(speaker_turn_counts), 2), 'speaker_turns_p95': percentile(speaker_turn_counts, .95)}

{'meetings': 1126,
 'characters_mean': 39126.02,
 'characters_median': 34036.0,
 'characters_p90': 72137,
 'characters_p95': 89675,
 'speaker_turns_mean': 297.3,
 'speaker_turns_p95': 741}